# Foundation 05 — Authorization, Approval, and Least Privilege

Derive least-privilege authority from authenticated identity and current application state, obtain a deterministic policy decision, issue independent exact approval for a high-risk action, and atomically reauthorize and consume it at the effect boundary.

![Authorization, approval, and atomic enforcement](architecture.svg)

The model proposes. Trusted identity, resource state, policy, an independent approver, and a final policy enforcement point govern whether one bounded refund effect occurs.

## 1. Load the course lab

The notebook imports the reusable course module rather than copying its security logic.

In [ ]:
import runpy
from datetime import datetime, timedelta, timezone
from dataclasses import replace
import sys
sys.path.insert(0, '.')
ns = runpy.run_path('lab.py')
ActionProposal, DecisionStatus = ns['ActionProposal'], ns['DecisionStatus']
build_scenario, refund_proposal = ns['build_scenario'], ns['refund_proposal']
authorize_and_approve = ns['authorize_and_approve']
now = datetime(2026,9,27,12,0,tzinfo=timezone.utc)
pep, actor, approver, cases = build_scenario()

## 2. Establish the safe baseline

Observe the trusted inputs and the decision evidence before injecting failures.

In [ ]:
direct = refund_proposal('notebook:direct',500)
direct_result = pep.enforce(actor,direct,now=now)
high = refund_proposal('notebook:approved',2500)
preflight = pep.pdp.evaluate(actor,high)
receipt = authorize_and_approve(pep,actor,approver,high,approval_id='approval:notebook',now=now)
approved_result = pep.enforce(actor,high,approval_id=receipt.approval_id,now=now)
assert direct_result.reason == 'direct-limit-effect'
assert preflight.status is DecisionStatus.PAUSE and approved_result.reason == 'approved-effect'
{'direct':direct_result,'high_risk_preflight':preflight,'approved':approved_result}

## 3. Inject an attack

Change one security-relevant boundary and keep the rest of the fixture stable.

In [ ]:
unprivileged = replace(actor,roles=frozenset(),scopes=frozenset())
claimed = refund_proposal('notebook:claimed-role',500,model_claimed_role='admin')
baseline_accepts = ns['unsafe_role_only_authorization'](unprivileged,claimed)
controlled = pep.enforce(unprivileged,claimed,now=now)
assert baseline_accepts and controlled.status is DecisionStatus.DENY and controlled.reason == 'scope'
{'role_only_baseline':baseline_accepts,'controlled':controlled}

## 4. Attempt a bypass

The assertions below make the security property executable and regression-testable.

In [ ]:
pep2,actor2,approver2,_ = build_scenario()
original = refund_proposal('notebook:binding',2500)
bound = authorize_and_approve(pep2,actor2,approver2,original,approval_id='approval:binding',now=now)
altered = replace(original,amount_cents=4000)
denied = pep2.enforce(actor2,altered,approval_id=bound.approval_id,now=now)
allowed = pep2.enforce(actor2,original,approval_id=bound.approval_id,now=now)
replay = pep2.enforce(actor2,original,approval_id=bound.approval_id,now=now)
assert denied.reason == 'approval-binding' and allowed.effect_applied and replay.reason == 'operation-replayed'
(denied,allowed,replay)

## 5. Evaluate observable outcomes

Use explicit denominators or counts. Private model reasoning is neither required nor recorded.

In [ ]:
report,evaluation_cases = ns['evaluate_controls'](now=now)
assert (report.cases,report.valid_cases,report.attack_cases,report.failure_cases) == (14,4,8,2)
assert report.attack_block_rate == report.valid_task_success_rate == 1
assert report.attack_effect_rate == report.blocked_valid_task_rate == 0
{'populations':{'all':report.cases,'valid':report.valid_cases,'attack':report.attack_cases,'failure':report.failure_cases},'rates':{'attack_block':report.attack_block_rate,'attack_effect':report.attack_effect_rate,'valid_success':report.valid_task_success_rate,'blocked_valid':report.blocked_valid_task_rate,'baseline_attack_acceptance':report.baseline_attack_acceptance_rate,'trace_completeness':report.trace_completeness_rate,'exposure_reduction':report.least_privilege_exposure_reduction},'outcomes':{case.name:(case.decision.status.value,case.decision.reason) for case in evaluation_cases}}

## 6. Exercise a second failure mode

In [ ]:
failed_pep,failed_actor,_,_ = build_scenario()
failed_pep.pdp.available = False
pdp_error = failed_pep.enforce(failed_actor,refund_proposal('notebook:pdp-error',500),now=now)
failed_pep.pdp.available = True
failed_pep.approvals.available = False
store_error = failed_pep.enforce(failed_actor,refund_proposal('notebook:store-error',2500),now=now)
assert pdp_error.status is store_error.status is DecisionStatus.ERROR
assert not failed_pep.effects
(pdp_error,store_error)

## 7. Least-privilege discovery is not enforcement

Derive the visible tool set from trusted actor and resource state. Then remember that a hidden tool can still be called directly, so the PEP must enforce every operation.

In [ ]:
visible = ns['minimum_tool_set'](actor,cases['case:north:42'])
none = ns['minimum_tool_set'](replace(actor,authenticated=False),cases['case:north:42'])
assert 'admin_refunds' not in visible and not none
{'catalog':sorted(ns['CATALOG']),'visible':sorted(visible),'exposure_reduction':(len(ns['CATALOG'])-len(visible))/len(ns['CATALOG'])}

## 8. Expiry, revocation, policy, and resource freshness

Approval records a review decision; it does not freeze the world. The PEP compares the receipt with current trusted state immediately before the effect.

In [ ]:
stale_pep,stale_actor,stale_approver,stale_cases = build_scenario()
stale_proposal = refund_proposal('notebook:stale',2500)
stale_receipt = authorize_and_approve(stale_pep,stale_actor,stale_approver,stale_proposal,approval_id='approval:stale',now=now)
stale_cases[stale_proposal.resource_id].version += 1
stale_result = stale_pep.enforce(stale_actor,stale_proposal,approval_id=stale_receipt.approval_id,now=now)
assert stale_result.reason == 'approval-stale' and not stale_result.effect_applied
stale_result

## 9. Atomic single use under concurrency

Eight workers race one exact receipt. Current authorization, receipt validation, effect creation, and consumption share one in-process critical section.

In [ ]:
allowed,denied,effects = ns['concurrency_probe'](workers=8,now=now)
assert (allowed,denied,effects) == (1,7,1)
{'allowed':allowed,'denied':denied,'effects':effects}

## 10. OpenAI Agents SDK dynamic approval hook

The real SDK pauses the high-risk call using the same PDP. The application then issues an exact receipt and routes final execution through the same PEP; no model or network call is made.

In [ ]:
sdk = runpy.run_path('sdk_adapter.py')
sdk_result,evidence = await sdk['credential_free_demo'](now=now)
assert sdk_result.effect_applied and evidence['paused'] and evidence['pdp_status'] == 'pause'
assert evidence['strict_json_schema'] and evidence['needs_approval_is_callable']
{'decision':sdk_result,'fields':sorted(evidence['input_schema']['properties']),'hook':{'paused':evidence['paused'],'pdp_status':evidence['pdp_status']}}

## 11. Production replacement

Production replacement: enterprise human and workload identity; audience-bound credentials; trusted current identity, relationship, resource, session, risk, and business-state inputs; reviewed versioned policy; authenticated embedded or remote PDP evaluation; explicit allow, pause, deny, error, diagnostic, and obligation semantics; dynamic discovery plus unconditional PEP checks; durable independent approval with exact consequence display, expiry, and revocation; transactional single-use and operation identity; provider idempotency and unknown-outcome reconciliation; privacy-aware tamper-evident decision evidence; and continuous policy, mutation, integration, concurrency, chaos, adversarial, and valid-task tests. The local dictionaries and lock prove only the in-process control sequence.

## 12. Exercises

1. Add queue membership as a relationship and prove role alone is insufficient.
2. Add an atomic cumulative refund budget.
3. Map the request to AuthZEN 1.0 while preserving local PAUSE and ERROR semantics.
4. Implement equivalent Cedar or OPA policy tests.
5. Model the resource hierarchy in OpenFGA.
6. Replace the process lock with a durable uniqueness constraint and inject crashes around provider commit.

## Checkpoint

Explain which trusted component enforces the invariant, what evidence proves the decision, and what residual risk remains.